In [ ]:
import tempfile
import os
from ansys.aedt.core import Desktop, generate_unique_name
import optuna
from optuna.samplers import GPSampler
import sys
import numpy as np
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..'))) 
import script_otim as script
import logging

c:\Users\luis.costa\.conda\envs\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
optuna.logging.get_logger("optuna").addHandler(logging.StreamHandler(sys.stdout))
study_name = "gp_seed1" 
storage_name = "sqlite:///{}.db".format(study_name)

In [3]:
tmpfold = tempfile.gettempdir()

temp_folder = os.path.join(
    tmpfold,
    generate_unique_name("Antenna")
)

if not os.path.exists(temp_folder):
    os.mkdir(temp_folder)

non_graphical = os.getenv(
    "PYAEDT_NON_GRAPHICAL",
    "True"
).lower() in ("true", "1", "t")

d = Desktop(
    "2025.1",
    non_graphical=non_graphical,
    new_desktop=True
)


PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].


PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\LUIS~1.COS\AppData\Local\Temp\pyaedt_luis.costa_f6eb4ef6-d130-4811-b6db-4298bbc177e7.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: New AEDT session is starting on gRPC port 54849.
PyAEDT INFO: Electronics Desktop started on gRPC port: 54849 after 28.171009063720703 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v251\AnsysEM
PyAEDT INFO: Ansoft.ElectronicsDesktop.2025.1 version started with process ID 9084.


In [ ]:
def objective(trial):
    a1 = trial.suggest_float("a1", 1.0, 1.0)
    a2 = trial.suggest_float("a2", 1.0, 1.0)
    n1 = trial.suggest_int("n1", 1, 100)
    n2 = trial.suggest_float("n2", 1.0, 10.0)
    n3 = trial.suggest_float("n3", 1.0, 10.0)
    n4 = trial.suggest_int("n4", 1, 100)
    perc = trial.suggest_float("perc", 0.2, 0.8)
    r = trial.suggest_int("r", 90, 150)

    df = script.makeAntenna(a1, a2, n1, n2, n3, n4, perc, r, temp_folder)
    df_filtrado = df[(df['Frequency [MHz]'] >= 0.3) & (df['Frequency [MHz]'] <= 1.5)]
    
    s11 = np.asarray(df_filtrado['|$S_{11}$| [dB]'], dtype=float)

    proportion_bad = float(np.mean(s11 >= -10))

    return proportion_bad

study = optuna.create_study(
    directions=["minimize"],    
    study_name=study_name, 
    storage=storage_name,
    sampler=GPSampler(seed=123),
    load_if_exists=True
)

    

[I 2025-12-11 13:33:35,981] Using an existing study with name 'tpe_seed1' instead of creating a new one.


Using an existing study with name 'tpe_seed1' instead of creating a new one.


In [ ]:
study.optimize(objective, n_trials=1000)

PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 9084!
PyAEDT INFO: Project Project29 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_E3E' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 4sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Deleted 1 Objects: cut.
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0

[I 2025-12-11 14:07:56,623] Trial 22 finished with value: 0.6859504132231405 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 90, 'n2': 7.991906990629111, 'n3': 3.788829553236882, 'n4': 13, 'perc': 0.7983204724668239, 'r': 103}. Best is trial 22 with value: 0.6859504132231405.


Trial 22 finished with value: 0.6859504132231405 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 90, 'n2': 7.991906990629111, 'n3': 3.788829553236882, 'n4': 13, 'perc': 0.7983204724668239, 'r': 103}. Best is trial 22 with value: 0.6859504132231405.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 9084!
PyAEDT INFO: Project Project30 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_AI9' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Mode

[I 2025-12-11 14:53:15,721] Trial 23 finished with value: 0.628099173553719 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 95, 'n2': 8.119533300186136, 'n3': 1.8565185274705558, 'n4': 14, 'perc': 0.7988479230651637, 'r': 96}. Best is trial 23 with value: 0.628099173553719.


Trial 23 finished with value: 0.628099173553719 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 95, 'n2': 8.119533300186136, 'n3': 1.8565185274705558, 'n4': 14, 'perc': 0.7988479230651637, 'r': 96}. Best is trial 23 with value: 0.628099173553719.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 9084!
PyAEDT INFO: Project Project31 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_1K2' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modele

...

Trial 425 finished with value: 0.049586776859504134 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 28, 'n2': 8.450909268958057, 'n3': 1.5770948268805136, 'n4': 39, 'perc': 0.5897815444667054, 'r': 98}. Best is trial 146 with value: 0.04132231404958678.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 9084!
PyAEDT INFO: Project Project433 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_L2N' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3

[I 2025-12-22 08:23:07,996] Trial 426 finished with value: 0.049586776859504134 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 29, 'n2': 7.787352766947376, 'n3': 1.329578194060064, 'n4': 41, 'perc': 0.5768411319073835, 'r': 100}. Best is trial 146 with value: 0.04132231404958678.


Trial 426 finished with value: 0.049586776859504134 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 29, 'n2': 7.787352766947376, 'n3': 1.329578194060064, 'n4': 41, 'perc': 0.5768411319073835, 'r': 100}. Best is trial 146 with value: 0.04132231404958678.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 9084!
PyAEDT INFO: Project Project434 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_KQL' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3

[I 2025-12-22 08:50:55,960] Trial 427 finished with value: 0.049586776859504134 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 26, 'n2': 7.971988660180464, 'n3': 1.4682475499936638, 'n4': 39, 'perc': 0.5985332770140491, 'r': 97}. Best is trial 146 with value: 0.04132231404958678.


Trial 427 finished with value: 0.049586776859504134 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 26, 'n2': 7.971988660180464, 'n3': 1.4682475499936638, 'n4': 39, 'perc': 0.5985332770140491, 'r': 97}. Best is trial 146 with value: 0.04132231404958678.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 9084!
PyAEDT INFO: Project Project435 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_KYV' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3

In [ ]:
len(study.trials)

6

In [ ]:
study.best_params

{'a1': 1.0,
 'a2': 1.0,
 'n1': 70,
 'n2': 3.575254014553415,
 'n3': 3.041663082077828,
 'n4': 56,
 'perc': 0.6316813818713379,
 'r': 115}

In [ ]:
study.best_trial

FrozenTrial(number=3, state=TrialState.COMPLETE, values=[0.8677685950413223], datetime_start=datetime.datetime(2025, 12, 10, 15, 0, 58, 189136), datetime_complete=datetime.datetime(2025, 12, 10, 15, 26, 55, 168417), params={'a1': 1.0, 'a2': 1.0, 'n1': 70, 'n2': 3.575254014553415, 'n3': 3.041663082077828, 'n4': 56, 'perc': 0.6316813818713379, 'r': 115}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'a1': FloatDistribution(high=1.0, log=False, low=1.0, step=None), 'a2': FloatDistribution(high=1.0, log=False, low=1.0, step=None), 'n1': IntDistribution(high=100, log=False, low=1, step=1), 'n2': FloatDistribution(high=10.0, log=False, low=1.0, step=None), 'n3': FloatDistribution(high=10.0, log=False, low=1.0, step=None), 'n4': IntDistribution(high=100, log=False, low=1, step=1), 'perc': FloatDistribution(high=0.8, log=False, low=0.2, step=None), 'r': IntDistribution(high=150, log=False, low=90, step=1)}, trial_id=4, value=None)

In [ ]:
d.release_desktop()

PyAEDT INFO: Desktop has been released and closed.


True